In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from dotenv import load_dotenv
from IPython.display import display,Markdown
import os
from langchain_core.messages import SystemMessage,HumanMessage



In [8]:
import langchain
print(langchain.__version__)

0.3.26


In [2]:
from openai import OpenAI
load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
openai = OpenAI()


DB_NAME="vect_db"

### load the pdf 

In [ ]:

loader=PyPDFLoader("/Users/dhanushreehm/project/llm_engineering/week5/NIPS-2017-attention-is-all-you-need-Paper.pdf")
docs=loader.load()



In [11]:
print(type(docs))
print(len(docs))
print(docs[0].page_content[:200])
print(docs[0].metadata)

<class 'list'>
11
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz
{'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring

### chunk it 

In [12]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=300)
chunks=splitter.split_documents(docs)

In [13]:
chunks[0]


Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On E

### vector embeddings -encoding llm

In [14]:
embeddings=HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
# embeddings=OpenAIEmbeddings(model="text-embedding-3-small")


In [15]:
vector=embeddings.embed_documents([chunk.page_content for chunk in chunks])
#this vector variable isnt needed below anymore because from_documents creates this already


In [16]:
if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME,embedding_function=embeddings).delete_collection()


vector_store=Chroma.from_documents(documents=chunks,embedding=embeddings,persist_directory=DB_NAME)

In [17]:
data=vector_store._collection.get(limit=1)
data.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [18]:
docs=vector_store.similarity_search_with_score("Why is the Transformer better than RNNs?",
                                   k=3)


In [19]:
#turning it into a retriever
retriever=vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":10,"fetch_k":40}
)
docs=retriever.invoke("advantages of self-attention over recurrent layers")
for doc in docs:
    print(doc.metadata["page"])

5
5
8
4
9
1
3
6
9
7


In [20]:

llm=ChatOpenAI(model="gpt-4.1-nano",temperature=0)

In [27]:
SYSTEM_PROMPT_TEMPLATE = """You are a technical research assistant. You answer questions about the
documents provided in the context below.

Rules:
1. Base every statement on the context. Do not use outside knowledge, even
   if you know the answer.
2. You may combine information from several passages, and you may draw
   conclusions that follow directly from what the passages say. Do not add
   claims the passages do not support.
3. You may identify a limitation, challenge, drawback, or area for improvement
when it is directly supported by the evidence, even if the document does not
explicitly label it as a "limitation."

Clearly distinguish between:
- what the authors explicitly state,
- what the evidence directly suggests,
- and what is not supported by the documents.

When making an interpretation, use wording such as "This suggests..." or
"The paper identifies this as an area for improvement..." rather than presenting
the interpretation as an explicit statement from the authors.
4. If the context only partly answers the question, say what the documents
   do support and state clearly what is missing. Do not guess the rest.
5. If the context has nothing relevant to the question, say that the
   available documents do not contain this information.
6. Do not mention "the context" or "the passages" unless you are saying
   something is missing. Just answer.

CONTEXT:
----------------
{context}
----------------"""

query_rewriting_prompt=""" You are a query rewriter for a document search system. Your only job is to
turn the user's latest question into a standalone search query.

You will receive a conversation history and the latest question.

Rules:
1. If the latest question depends on the history to be understood (pronouns
   like "it", "they", "that", or references like "the second point"),
   rewrite it so it is fully understandable on its own. Replace each
   reference with the specific thing it refers to.
2. If the latest question is already understandable on its own, return it
   exactly as written, word for word.
3. Never answer the question.
4. Never add information that is not in the conversation or the question.
5. Do not explain what you did.
6. Output only the final question as plain text: no quotes, no labels, no
   preamble.   

Example 1
History:
user: What is the Transformer?
assistant: The Transformer is a model architecture based entirely on attention mechanisms.
latest question: Why is it better than RNNs?
output: Why is the Transformer better than RNNs?

Example 2
History:
user: What is the Transformer?
assistant: The Transformer is a model architecture based entirely on attention mechanisms.
latest question: What BLEU score did the big model achieve on English-to-German?
output: What BLEU score did the big model achieve on English-to-German?"""


def query_rewrite(latest_question:str,chat_history):
    qa_history=[f"{h['role']} : {h['content']}" for h in chat_history ]
    qs="\n".join(qa_history[-8:])+" \n Latest question :"+latest_question
    rewritten_q=(llm.invoke([SystemMessage(content=query_rewriting_prompt),HumanMessage(content=qs)])).content
    display(Markdown(rewritten_q))
    docs=retriever.invoke(rewritten_q)
    sources=[{"source":doc.metadata["title"],"page":doc.metadata["page"],"content":doc.page_content} for doc in docs ]
    context = "\n\n".join(f"[p.{doc.metadata['page']}] {doc.page_content}" for doc in docs)
    return rewritten_q,context,sources
    
def chat(question:str,history):
    updated_q,context,sources =query_rewrite(question,history)
    display(Markdown(context))
    system_prompt=SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response=llm.invoke([SystemMessage(content=system_prompt),HumanMessage(content=updated_q)])
    return response.content 


In [26]:
import gradio as gr
gr.ChatInterface(fn=chat,type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


What are the limitations of transformer models

[p.8] plan to extend the Transformer to problems involving input and output modalities other than text and
to investigate local, restricted attention mechanisms to efﬁciently handle large inputs and outputs
such as images, audio and video. Making generation less sequential is another research goals of ours.
The code we used to train and evaluate our models is available at https://github.com/
tensorflow/tensor2tensor.
Acknowledgements We are grateful to Nal Kalchbrenner and Stephan Gouws for their fruitful
comments, corrections and inspiration.
9

[p.7] architectures from the literature. We estimate the number of ﬂoating point operations used to train a
model by multiplying the training time, the number of GPUs used, and an estimate of the sustained
single-precision ﬂoating-point capacity of each GPU 5.
6.2 Model Variations
To evaluate the importance of different components of the Transformer, we varied our base model
in different ways, measuring the change in performance on English-to-German translation on the
development set, newstest2013. We used beam search as described in the previous section, but no
checkpoint averaging. We present these results in Table 3.
In Table 3 rows (A), we vary the number of attention heads and the attention key and value dimensions,
keeping the amount of computation constant, as described in Section 3.2.2. While single-head
attention is 0.9 BLEU worse than the best setting, quality also drops off with too many heads.

[p.7] Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the
English-to-German and English-to-French newstest2014 tests at a fraction of the training cost.
Model
BLEU Training Cost (FLOPs)
EN-DE EN-FR EN-DE EN-FR
ByteNet [15] 23.75
Deep-Att + PosUnk [32] 39.2 1.0· 1020
GNMT + RL [31] 24.6 39.92 2.3· 1019 1.4· 1020
ConvS2S [8] 25.16 40.46 9.6· 1018 1.5· 1020
MoE [26] 26.03 40.56 2.0· 1019 1.2· 1020
Deep-Att + PosUnk Ensemble [32] 40.4 8.0· 1020
GNMT + RL Ensemble [31] 26.30 41.16 1.8· 1020 1.1· 1021
ConvS2S Ensemble [8] 26.36 41.29 7.7· 1019 1.2· 1021
Transformer (base model) 27.3 38.1 3.3 · 1018
Transformer (big) 28.4 41.0 2.3· 1019
Label Smoothing During training, we employed label smoothing of value ϵls = 0.1 [30]. This
hurts perplexity, as the model learns to be more unsure, but improves accuracy and BLEU score.
6 Results
6.1 Machine Translation
On the WMT 2014 English-to-German translation task, the big transformer model (Transformer (big)

[p.1] of continuous representations z = (z1,...,z n). Given z, the decoder then generates an output
sequence (y1,...,y m) of symbols one element at a time. At each step the model is auto-regressive
[9], consuming the previously generated symbols as additional input when generating the next.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The ﬁrst is a multi-head self-attention mechanism, and the second is a simple, position-
2

[p.7] outperforming all of the previously published single models, at less than 1/4 the training cost of the
previous state-of-the-art model. The Transformer (big) model trained for English-to-French used
dropout ratePdrop = 0.1, instead of 0.3.
For the base models, we used a single model obtained by averaging the last 5 checkpoints, which
were written at 10-minute intervals. For the big models, we averaged the last 20 checkpoints. We
used beam search with a beam size of 4 and length penaltyα = 0.6 [31]. These hyperparameters
were chosen after experimentation on the development set. We set the maximum output length during
inference to input length + 50, but terminate early when possible [31].
Table 2 summarizes our results and compares our translation quality and training costs to other model
architectures from the literature. We estimate the number of ﬂoating point operations used to train a
model by multiplying the training time, the number of GPUs used, and an estimate of the sustained

[p.8] 1024 5.12 25.4 53
4096 4.75 26.2 90
(D)
0.0 5.77 24.6
0.2 4.95 25.5
0.0 4.67 25.3
0.2 5.47 25.7
(E) positional embedding instead of sinusoids 4.92 25.7
big 6 1024 4096 16 0.3 300K 4.33 26.4 213
In Table 3 rows (B), we observe that reducing the attention key size dk hurts model quality. This
suggests that determining compatibility is not easy and that a more sophisticated compatibility
function than dot product may be beneﬁcial. We further observe in rows (C) and (D) that, as expected,
bigger models are better, and dropout is very helpful in avoiding over-ﬁtting. In row (E) we replace our
sinusoidal positional encoding with learned positional embeddings [8], and observe nearly identical
results to the base model.
7 Conclusion
In this work, we presented the Transformer, the ﬁrst sequence transduction model based entirely on
attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with
multi-headed self-attention.

[p.8] Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base
model. All metrics are on the English-to-German translation development set, newstest2013. Listed
perplexities are per-wordpiece, according to our byte-pair encoding, and should not be compared to
per-word perplexities.
N d model dff h d k dv Pdrop ϵls
train PPL BLEU params
steps (dev) (dev) ×106
base 6 512 2048 8 64 64 0.1 0.1 100K 4.92 25.8 65
(A)
1 512 512 5.29 24.9
4 128 128 5.00 25.5
16 32 32 4.91 25.8
32 16 16 5.01 25.4
(B) 16 5.16 25.1 58
32 5.01 25.4 60
(C)
2 6.11 23.7 36
4 5.19 25.3 50
8 4.88 25.5 80
256 32 32 5.75 24.5 28
1024 128 128 4.66 26.0 168
1024 5.12 25.4 53
4096 4.75 26.2 90
(D)
0.0 5.77 24.6
0.2 4.95 25.5
0.0 4.67 25.3
0.2 5.47 25.7
(E) positional embedding instead of sinusoids 4.92 25.7
big 6 1024 4096 16 0.3 300K 4.33 26.4 213
In Table 3 rows (B), we observe that reducing the attention key size dk hurts model quality. This

[p.4] MultiHead(Q,K,V ) = Concat(head 1,..., headh)W O
where headi = Attention(QW Q
i ,KW K
i ,VW V
i )
Where the projections are parameter matricesW Q
i ∈ Rdmodel×dk,W K
i ∈ Rdmodel×dk,W V
i ∈ Rdmodel×dv
andW O∈ Rhdv×dmodel.
In this work we employ h = 8 parallel attention layers, or heads. For each of these we use
dk =dv =dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost
is similar to that of single-head attention with full dimensionality.
3.2.3 Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[31, 2, 8].

[p.0] Recurrent neural networks, long short-term memory [12] and gated recurrent [7] neural networks
in particular, have been ﬁrmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [ 29, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures [31, 21, 13].
∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started
the effort to evaluate this idea. Ashish, with Illia, designed and implemented the ﬁrst Transformer models and
has been crucially involved in every aspect of this work. Noam proposed scaled dot-product attention, multi-head
attention and the parameter-free position representation and became the other person involved in nearly every
detail. Niki designed, implemented, tuned and evaluated countless model variants in our original codebase and

[p.9] arXiv:1308.0850, 2013.
[10] Kaiming He, Xiangyu Zhang, Shaoqing Ren, and Jian Sun. Deep residual learning for im-
age recognition. In Proceedings of the IEEE Conference on Computer Vision and Pattern
Recognition, pages 770–778, 2016.
[11] Sepp Hochreiter, Yoshua Bengio, Paolo Frasconi, and Jürgen Schmidhuber. Gradient ﬂow in
recurrent nets: the difﬁculty of learning long-term dependencies, 2001.
[12] Sepp Hochreiter and Jürgen Schmidhuber. Long short-term memory. Neural computation,
9(8):1735–1780, 1997.
[13] Rafal Jozefowicz, Oriol Vinyals, Mike Schuster, Noam Shazeer, and Yonghui Wu. Exploring
the limits of language modeling. arXiv preprint arXiv:1602.02410, 2016.
[14] Łukasz Kaiser and Ilya Sutskever. Neural GPUs learn algorithms. In International Conference
on Learning Representations (ICLR), 2016.
[15] Nal Kalchbrenner, Lasse Espeholt, Karen Simonyan, Aaron van den Oord, Alex Graves, and Ko-

In [31]:
# Paste into a notebook cell AFTER your cell that defines
# SYSTEM_PROMPT_TEMPLATE, query_rewriting_prompt, query_rewrite and chat.
# pip install -U gradio

import html
import gradio as gr
from langchain_core.messages import SystemMessage, HumanMessage


def _text(content):
    """Some Gradio versions give message content as a list of blocks. Always return a string."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return " ".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in content)
    return str(content)


def clean_history(history):
    return [{"role": h["role"], "content": _text(h["content"])} for h in (history or [])]


def render_sources(sources, rewritten=None):
    if not sources:
        return "<p style='opacity:.6'>Sources will appear here after you ask a question.</p>"
    parts = []
    if rewritten:
        parts.append(f"<p style='font-size:13px;opacity:.7'><b>Searched for:</b> {html.escape(rewritten)}</p>")
    for i, s in enumerate(sources, 1):
        parts.append(
            "<details style='margin-bottom:8px;border:1px solid #8884;border-radius:8px;padding:8px 10px'>"
            f"<summary style='cursor:pointer'><b>{i}. {html.escape(str(s['source']))}</b> · page {html.escape(str(s['page']))}</summary>"
            f"<div style='margin-top:6px;font-size:13px;white-space:pre-wrap'>{html.escape(s['content'])}</div>"
            "</details>"
        )
    # wrap everything in a fixed-height scrollable box
    return "<div style='max-height:350px;overflow-y:auto;padding-right:6px'>" + "".join(parts) + "</div>"


def respond(message, history):
    message = (message or "").strip()
    history = clean_history(history)
    if not message:
        return "", history, gr.update()
    try:
        # same steps as your chat(), but we keep the sources so we can show them
        rewritten_q, context, sources = query_rewrite(message, history)
        system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
        answer = llm.invoke([SystemMessage(content=system_prompt),
                             HumanMessage(content=rewritten_q)]).content
        sources_html = render_sources(sources, rewritten_q)
    except Exception as e:
        answer = f"Error: {e}"
        sources_html = "<p>Something went wrong while retrieving sources.</p>"
    history = history + [{"role": "user", "content": message},
                         {"role": "assistant", "content": answer}]
    return "", history, sources_html


def clear():
    return "", [], render_sources([])


with gr.Blocks(title="Paper Q&A") as demo:
    gr.Markdown("## Research Paper Assistant")
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=350, label="Chat",type="messages")
            msg = gr.Textbox(placeholder="Ask about the paper and press Enter...", show_label=False)
            with gr.Row():
                send = gr.Button("Send", variant="primary")
                clear_btn = gr.Button("Clear")
        with gr.Column(scale=2):
            gr.Markdown("### Sources")
            sources_panel = gr.HTML(render_sources([]))

    msg.submit(respond, [msg, chatbot], [msg, chatbot, sources_panel])
    send.click(respond, [msg, chatbot], [msg, chatbot, sources_panel])
    clear_btn.click(clear, None, [msg, chatbot, sources_panel])

demo.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


What is a transformer

What are self-attention layers in Transformer models

What are self-attention layers in transformer models